# NB53: Spark + Elasticsearch

Real-time log analysis indexing to Elasticsearch.

## 1. Environment Setup

This cell installs **Java 8**, **Spark 3.5.0**, **Kafka 3.6.1**, and necessary Python libraries (`pyspark`, `kafka-python`, `redis`, `pymongo`, `elasticsearch`, `cassandra-driver`, `minio`). It also sets environment variables for Java and Spark.

In [ ]:
# Install Dependencies
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!wget -q https://archive.apache.org/dist/kafka/3.6.1/kafka_2.13-3.6.1.tgz
!tar xf kafka_2.13-3.6.1.tgz
!pip install -q findspark pyspark kafka-python redis pymongo elasticsearch==7.10.1 cassandra-driver minio "numpy<2.0.0"

# Environment Variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"
import findspark
findspark.init()

## 2. Start Services

This cell starts the required distributed services in the background:
- **Kafka & Zookeeper**: Event streaming platform.
- **Elasticsearch**: Search and analytics engine.

In [ ]:
# Start Kafka
!./kafka_2.13-3.6.1/bin/zookeeper-server-start.sh -daemon ./kafka_2.13-3.6.1/config/zookeeper.properties
!./kafka_2.13-3.6.1/bin/kafka-server-start.sh -daemon ./kafka_2.13-3.6.1/config/server.properties
# Start Elasticsearch
!wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-7.10.2-linux-x86_64.tar.gz
!tar -xzf elasticsearch-7.10.2-linux-x86_64.tar.gz
!chown -R daemon:daemon elasticsearch-7.10.2
!sudo -u daemon ES_JAVA_OPTS="-Xms512m -Xmx512m" ./elasticsearch-7.10.2/bin/elasticsearch -d > es.log 2>&1 &

import time, socket, os
def wait_for_port(port, host='localhost', timeout=120):
    start_time = time.time()
    while True:
        try:
            with socket.create_connection((host, port), timeout=1):
                print(f"Service at {host}:{port} is ready!")
                return True
        except (OSError, ConnectionRefusedError):
            if time.time() - start_time > timeout:
                print(f"Timeout waiting for {host}:{port} to start.")
                # Dump logs for debugging
                if os.path.exists('minio.log'):
                    print('--- MINIO LOG ---')
                    print(open('minio.log').read())
                if os.path.exists('es.log'):
                    print('--- ES LOG ---')
                    print(open('es.log').read())
                if os.path.exists('cassandra.log'):
                    print('--- CASSANDRA LOG ---')
                    print(open('cassandra.log').read())
                raise Exception(f"Service at {host}:{port} failed to start.")
            time.sleep(2)

# Wait for services
wait_for_port(9092) # Kafka
wait_for_port(9200) # Elasticsearch


## 3. Create Kafka Topic

Creates a topic named `input-topic` with 1 partition and replication factor 1.

In [ ]:
# Create Topic
!./kafka_2.13-3.6.1/bin/kafka-topics.sh --create --topic input-topic --bootstrap-server localhost:9092 --replication-factor 1 --partitions 1

## 4. Producer

Generates log messages (`INFO` or `ERROR`) and sends them to Kafka.

In [ ]:
from kafka import KafkaProducer
import json, time, random
print("Starting Log Producer...")
producer = KafkaProducer(bootstrap_servers='localhost:9092')
print("Sending 100 log messages...")
for _ in range(100):
    log = {'timestamp': time.time(), 'level': random.choice(['INFO', 'ERROR'])}
    producer.send('input-topic', json.dumps(log).encode('utf-8'))
producer.flush()
print("Producer finished.")

## 5. Spark -> Elasticsearch

Reads logs from Kafka and indexes them into Elasticsearch index `logs`.

In [ ]:
%%writefile kafka_consumer.py
from pyspark.sql import SparkSession
from elasticsearch import Elasticsearch
import json

spark = SparkSession.builder.appName("Elastic").getOrCreate()

def process_batch(df, epoch_id):
    rows = df.collect()
    es = Elasticsearch(['http://localhost:9200'])
    for row in rows:
        doc = json.loads(row.value)
        es.index(index='logs', body=doc)
    print(f"Batch {epoch_id} processed: {len(rows)} logs indexed.")

print("Starting Spark Streaming Job...")
df = spark.readStream.format("kafka").option("kafka.bootstrap.servers", "localhost:9092").option("subscribe", "input-topic").option("startingOffsets", "earliest").load()
query = df.selectExpr("CAST(value AS STRING)").writeStream.foreachBatch(process_batch).start()
query.awaitTermination(30)
print("Spark Job Finished.")

In [ ]:
!spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 kafka_consumer.py

## 6. Verification

Search Elasticsearch to verify indexed logs.

In [ ]:
from elasticsearch import Elasticsearch
es = Elasticsearch(['http://localhost:9200'])
time.sleep(2) # Wait for flush
print("Querying Elasticsearch...")
res = es.search(index="logs", body={"query": {"match_all": {}}, "size": 5})
print("--- Logs in Elasticsearch ---")
for hit in res['hits']['hits']:
    print(hit['_source'])